In [ ]:
import os
import glob
import json

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

pd.options.display.max_columns = None
pd.options.display.float_format = "{:,.4f}".format

from utils import CUSTOM_PALETTE, setup_plot_style

setup_plot_style()
pd.options.display.float_format = (
    "{:,.4f}".format
)  # restore 4-digit precision after setup
custom_palette = CUSTOM_PALETTE
%config InlineBackend.figure_format = 'retina'

In [ ]:
dir_path_base = os.path.expanduser("~/tunable-magmax")

model = "ViT-B-16"
dataset = "CIFAR100"
n_splits = 5
task_seq = "A"
lambda_ = 0.5
merge_fn = "merge_max_abs_masked_with_targetdata"
similarity_metric = "mmd_embedded"  # cosine_embedded ot_embedded labels

base_dir = (
    f"{dir_path_base}/logs/{model}/sequential_finetuning/"
    f"class_incremental/merging_target_data_cameraready/"
    f"{dataset}-{n_splits}/taskseq_{task_seq}/{merge_fn}/{similarity_metric}"
)

json_pattern = os.path.join(base_dir, "*.json")
json_files = sorted(glob.glob(json_pattern))
print(f"Found {len(json_files)} JSON files in:\n  {base_dir}")

In [ ]:
records = []

for fp in json_files:
    with open(fp, "r") as f:
        d = json.load(f)

    fname = os.path.basename(fp)
    num_unaligned = d.get("num_unaligned", {})
    num_params_all = d.get("num_params_all", None)
    total_unaligned = sum(num_unaligned.values())
    ratio = total_unaligned / num_params_all if num_params_all else None

    rec = {
        "file": fname,
        "num_params_all": num_params_all,
        "total_unaligned": total_unaligned,
        "ratio": ratio,
        "overall_accuracy": d.get("overall_accuracy"),
        "target_id": d.get("target_dataset_info", {}).get("target_id"),
        "seed": d.get("target_dataset_info", {}).get("seed_target_data"),
    }
    for task_key, val in num_unaligned.items():
        rec[f"unaligned_{task_key}"] = val

    records.append(rec)

df = pd.DataFrame(records)
print(f"Loaded {len(df)} records")
df.head()

In [ ]:
print("=== num_unaligned ===")
print(df["total_unaligned"].describe())
print()
print("=== num_params_all (ratio) ===")
print(df["ratio"].describe())
print()
print(f"num_params_all: {df['num_params_all'].iloc[0]:,}")
print(
    f"total_unaligned median: {df['total_unaligned'].median():,.0f}  ({df['ratio'].median() * 100:.2f}%)"
)
print(
    f"total_unaligned mean: {df['total_unaligned'].mean():,.0f}  ({df['ratio'].mean() * 100:.2f}%)"
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- histogram: total_unaligned ---
ax = axes[0]
sns.histplot(
    df["total_unaligned"],
    bins=20,
    kde=True,
    color=custom_palette[0],
    ax=ax,
)
ax.axvline(
    df["total_unaligned"].mean(),
    color=custom_palette[3],
    linestyle="--",
    label=f"Mean: {df['total_unaligned'].mean():,.0f}",
)
ax.axvline(
    df["total_unaligned"].median(),
    color=custom_palette[4],
    linestyle=":",
    label=f"Median: {df['total_unaligned'].median():,.0f}",
)
ax.set_xlabel("Total num_unaligned")
ax.set_ylabel("Count")
ax.set_title("Histogram of total num_unaligned")
ax.legend()

# --- histogram: ratio to num_params_all ---
ax = axes[1]
sns.histplot(
    df["ratio"] * 100,
    bins=20,
    kde=True,
    color=custom_palette[1],
    ax=ax,
)
ax.axvline(
    df["ratio"].mean() * 100,
    color=custom_palette[3],
    linestyle="--",
    label=f"Mean: {df['ratio'].mean() * 100:.2f}%",
)
ax.axvline(
    df["ratio"].median() * 100,
    color=custom_palette[4],
    linestyle=":",
    label=f"Median: {df['ratio'].median() * 100:.2f}%",
)
ax.set_xlabel("Ratio to num_params_all (%)")
ax.set_ylabel("Count")
ax.set_title("Histogram of num_unaligned / num_params_all (%)")
ax.legend()

plt.tight_layout()
save_path = base_dir.replace(similarity_metric, "visualise_num_unaligned")
os.makedirs(save_path, exist_ok=True)
plt.savefig(os.path.join(save_path, f"{similarity_metric}.pdf"))

In [ ]:
# task-wise num_unaligned
task_cols = [c for c in df.columns if c.startswith("unaligned_task_")]
n_tasks = len(task_cols)

if n_tasks > 0:
    fig, axes = plt.subplots(1, n_tasks, figsize=(4 * n_tasks, 5), sharey=False)
    if n_tasks == 1:
        axes = [axes]

    for ax, col in zip(axes, task_cols):
        task_label = col.replace("unaligned_", "")
        color = custom_palette[task_cols.index(col) % len(custom_palette)]
        sns.histplot(df[col], bins=15, kde=True, color=color, ax=ax)
        ax.set_xlabel("num_unaligned")
        ax.set_ylabel("Count")
        ax.set_title(task_label)

    plt.suptitle("Per-task num_unaligned distribution", y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# scatter plot: total_unaligned vs overall_accuracy
fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(
    df["total_unaligned"],
    df["overall_accuracy"],
    # c=df["target_id"],
    cmap="viridis",
    alpha=0.7,
    edgecolors="k",
    linewidths=0.4,
)
# plt.colorbar(sc, ax=ax, label="target_id")
ax.set_xlabel("Total num_unaligned")
ax.set_ylabel("Overall Accuracy")
ax.set_title("Total num_unaligned vs Overall Accuracy")
plt.tight_layout()

save_path = base_dir.replace(similarity_metric, "visualise_num_unaligned")
os.makedirs(save_path, exist_ok=True)
plt.savefig(os.path.join(save_path, f"{similarity_metric}_scatter.pdf"))
plt.show()